In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Provide path to input files

images_dir = Path(r'input_path')
images_path = os.path.join(images_dir,'*.stk') 
images_files = np.sort(glob.glob(images_path))

print(images_dir)
print(images_path)
print(len(images_files))

In [ ]:
# Split images into separate channels

images_files_halo = [f for f in images_files if 'conf561' in f]
images_files_tubulin = [f for f in images_files if 'conf488' in f]
images_files_actin = [f for f in images_files if 'conf640' in f]

In [ ]:
# Initialize lists to store images by channel

images_halo = []
images_tubulin = []
images_actin = []

# Sort images into respective lists based on filename cues
for file in tqdm(images_files):
    if 'conf561' in file:
        images_halo.append(imread(file))
    elif 'conf488' in file:
        images_tubulin.append(imread(file))
    elif 'conf640' in file:
        images_actin.append(imread(file))

# Check the lists
print(f"Halo images: {len(images_halo)}")
print(f"Tubulin images: {len(images_tubulin)}")
print(f"Actin images: {len(images_actin)}")

In [ ]:
np.unique(images_actin[0])

In [ ]:
# Function to create a max projection for slices 8 to 10
def max_projection(image, start_slice=7, end_slice=11):
    return np.max(image[start_slice:end_slice+1], axis=0)

# Apply max projection to each list
max_proj_halo = [max_projection(img) for img in images_halo]
max_proj_tubulin = [max_projection(img) for img in images_tubulin]
max_proj_actin = [max_projection(img) for img in images_actin]

# Check the number of max projections created
print(f"Max projections for Halo: {len(max_proj_halo)}")
print(f"Max projections for Tubulin: {len(max_proj_tubulin)}")
print(f"Max projections for Actin: {len(max_proj_actin)}")

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(max_proj_halo[2])
ax[1].imshow(max_proj_tubulin[2])
ax[2].imshow(max_proj_actin[2])

In [ ]:
# Save cytoplasm masks and corrected nuclei masks - ADJUST relative path names
# Please provide the path to the output directory

mask_cytoplasm_output_dir = Path(r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/Max_projections_seperate')

max_project_dir = mask_cytoplasm_output_dir


In [ ]:
# Save max projections to seperate folders

# Define the folder names
folders = ['Halo', 'Tubulin', 'Actin']

# Create the folders if they don't exist
for folder in folders:
    folder_path = os.path.join(max_project_dir, folder)
    os.makedirs(folder_path, exist_ok=True)

# Function to save a max projection image
def save_max_projection_image(image, original_file, channel_name):
    # Extract the file name (without path)
    file_name = os.path.basename(original_file)
    
    # Extract the base name (remove everything after the last '_')
    base_name = file_name.rsplit('_', 1)[0]  # This splits at the last underscore
    
    # Create the new filename with the appropriate suffix (e.g., "_halo", "_tubulin", "_actin")
    new_file_name = base_name + f"_{channel_name}.tif"
    
    # Define the output path
    output_file = os.path.join(max_project_dir, channel_name, new_file_name)
    
    # Check if file already exists to prevent overwriting
    if os.path.exists(output_file):
        print(f"Warning: {output_file} already exists! Skipping...")
        return
    
    # Save the max projection image to the corresponding folder
    tf.imwrite(output_file, image)
    print(f"Saved {output_file}")

# Now save the max projections for each channel
for i, image_file in enumerate(images_files_halo):
    save_max_projection_image(max_proj_halo[i], image_file, 'halo')

for i, image_file in enumerate(images_files_tubulin):
    save_max_projection_image(max_proj_tubulin[i], image_file, 'tubulin')

for i, image_file in enumerate(images_files_actin):
    save_max_projection_image(max_proj_actin[i], image_file, 'actin')